In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from scipy.optimize import minimize
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Define commodity tickers (removed duplicates)
tickers = list(set(['CL=F', 'GC=F', 'SI=F', 'HG=F', 'NG=F', 'BZ=F', 'ZC=F', 'ZS=F', 'ZW=F', 'CT=F',
                    'SB=F', 'KC=F', 'CC=F', 'OJ=F', 'PA=F', 'PL=F', 'HO=F', 'RB=F','BTC-USD']))

def download_data(tickers, start, end):
    data = yf.download(tickers, start=start, end=end, progress=False)['Close']
    returns = data.pct_change().dropna()
    return returns

def calculate_correlations(returns, lookback_years=2):
    lookback_days = int(lookback_years * 252)
    correlation_matrix = returns[-lookback_days:].corr()
    return correlation_matrix

def select_least_correlated(correlation_matrix, num_assets=10):
    avg_correlations = correlation_matrix.mean()
    least_correlated = avg_correlations.nsmallest(num_assets).index.tolist()
    return least_correlated

def portfolio_return(weights, mu):
    return np.sum(mu * weights)

def portfolio_volatility(weights, cov_matrix):
    return np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))

def mean_variance_optimization(returns, selected_tickers):
    mu = returns[selected_tickers].mean() * 252
    cov_matrix = returns[selected_tickers].cov() * 252
    num_assets = len(selected_tickers)
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1},
                   {'type': 'ineq', 'fun': lambda x: x})
    bounds = [(0, 1)] * num_assets
    initial = [1. / num_assets] * num_assets
    result = minimize(
        lambda x: -portfolio_return(x, mu) / portfolio_volatility(x, cov_matrix),
        initial,
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )
    return result.x, portfolio_return(result.x, mu), portfolio_volatility(result.x, cov_matrix)

def calculate_sharpe(returns, rf=0.0):
    ann_mean = returns.mean() * 252
    ann_std = returns.std() * np.sqrt(252)
    if ann_std == 0:
        return 0
    return (ann_mean - rf) / ann_std

def calculate_max_drawdown(cum_returns):
    if len(cum_returns) == 0:
        return 0
    roll_max = cum_returns.cummax()
    drawdown = (cum_returns / roll_max) - 1
    return drawdown.min()

def periodic_rebalance(tickers, start_date, end_date, rebalance_days=63, lookback_years=2):
    start_date = pd.to_datetime(start_date)
    end_date = pd.to_datetime(end_date)
    buffer_start = start_date - timedelta(days=int(lookback_years * 365 + 30))
    full_returns = download_data(tickers, buffer_start, end_date + timedelta(days=1))
    rebalance_dates = pd.date_range(start=start_date, end=end_date, freq=f'{rebalance_days}D').tolist()
    if end_date not in rebalance_dates:
        rebalance_dates.append(end_date)
    rebalance_dates = sorted(set([start_date] + [d for d in rebalance_dates if d > start_date]))
    strategy_returns = pd.Series(dtype=float)
    portfolio_results = []
    for i in range(len(rebalance_dates) - 1):
        rebalance_date = rebalance_dates[i]
        hold_end = rebalance_dates[i + 1]
        lookback_start = rebalance_date - timedelta(days=lookback_years * 365)
        lookback_returns = full_returns.loc[lookback_start:rebalance_date].copy()
        if lookback_returns.shape[0] < 100:
            continue
        correlation_matrix = lookback_returns.corr()
        selected_tickers = select_least_correlated(correlation_matrix)
        weights, exp_ret, vol = mean_variance_optimization(lookback_returns, selected_tickers)
        forward_idx = (full_returns.index > rebalance_date) & (full_returns.index <= hold_end)
        forward_returns = full_returns.loc[forward_idx, selected_tickers]
        if forward_returns.empty:
            continue
        port_daily_ret = (forward_returns * weights).sum(axis=1)
        strategy_returns = pd.concat([strategy_returns, port_daily_ret])
        portfolio_results.append({
            'date': rebalance_date,
            'tickers': selected_tickers,
            'weights': weights,
            'expected_return': exp_ret,
            'volatility': vol
        })
    # Cumulative returns
    if not strategy_returns.empty:
        strategy_cum = (1 + strategy_returns).cumprod()
        first_date = strategy_returns.index.min()
        asset_returns_period = full_returns.loc[full_returns.index >= first_date]
        assets_cum = (1 + asset_returns_period).cumprod()
        # Plot
        plt.figure(figsize=(14, 8))
        plt.plot(strategy_cum, label='Strategy', linewidth=3, color='black')
        for ticker in tickers:
            plt.plot(assets_cum[ticker], label=ticker, alpha=0.7, linewidth=1)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
        plt.title(f'Cumulative Returns of Strategy (Rebalance every {rebalance_days} days) and Individual Assets')
        plt.xlabel('Date')
        plt.ylabel('Cumulative Return')
        plt.grid(True)
        plt.tight_layout()
        plt.show()
    else:
        strategy_cum = pd.Series()
        print("No strategy returns calculated.")
    # Performance metrics
    strategy_sharpe = calculate_sharpe(strategy_returns)
    strategy_max_dd = calculate_max_drawdown(strategy_cum)
    print("\nStrategy Performance:")
    print(f"Sharpe Ratio: {strategy_sharpe:.2f}")
    print(f"Max Drawdown: {strategy_max_dd:.2%}")
    return portfolio_results

# Example usage
start_date = '2010-01-01'
end_date = datetime.now().strftime('%Y-%m-%d')
results = periodic_rebalance(tickers, start_date, end_date, rebalance_days=126)  # Approx. quarterly

# Print results
for result in results:
    print(f"\nRebalance Date: {result['date'].strftime('%Y-%m-%d')}")
    print("Selected Tickers:", result['tickers'])
    print("Weights:", [f"{w:.4f}" for w in result['weights']])
    print(f"Expected Return: {result['expected_return']:.4f}")
    print(f"Volatility: {result['volatility']:.4f}")